But : classif. supervisée (ex. mutant vs contrôle) avec PyTorch (MLP simple), + courbes ROC/PR, CV

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = pd.read_csv("../data/processed/features.csv")
Xn = torch.tensor(X.drop(columns=["path","morphotype","condition"]).fillna(0).values, dtype=torch.float32)
y = torch.tensor((X["condition"]=="mutant").astype(int).values, dtype=torch.float32)

Xtr,Xte,ytr,yte = train_test_split(Xn,y,test_size=0.2, stratify=y, random_state=42)

model = nn.Sequential(nn.Linear(Xtr.shape[1],64), nn.ReLU(), nn.Dropout(0.2),
                      nn.Linear(64,1))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.BCEWithLogitsLoss()

for epoch in range(200):
    opt.zero_grad(); loss = lossf(model(Xtr).squeeze(), ytr); loss.backward(); opt.step()

with torch.no_grad():
    proba = torch.sigmoid(model(Xte).squeeze()).numpy()
print("ROC AUC:", roc_auc_score(yte.numpy(), proba))
